In [1]:
from azure.ai.ml import MLClient, Input, Output
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import command
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import os

# Connect to workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

Found the config file in: /config.json


Connected: ml-learning-workspace


In [2]:
import os

# Create folder for pipeline scripts
os.makedirs("pipeline_scripts", exist_ok=True)

# Step 1 - Data preparation script
data_prep_script = '''
import argparse
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import os

parser = argparse.ArgumentParser()
parser.add_argument("--output_train", type=str)
parser.add_argument("--output_test", type=str)
args = parser.parse_args()

# Load and split data
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["target"] = iris.target

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save outputs
os.makedirs(args.output_train, exist_ok=True)
os.makedirs(args.output_test, exist_ok=True)
train_df.to_csv(os.path.join(args.output_train, "train.csv"), index=False)
test_df.to_csv(os.path.join(args.output_test, "test.csv"), index=False)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
'''

# Step 2 - Training script
train_script = '''
import argparse
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import mlflow
import joblib
import os

parser = argparse.ArgumentParser()
parser.add_argument("--input_train", type=str)
parser.add_argument("--output_model", type=str)
args = parser.parse_args()

# Load data
train_df = pd.read_csv(os.path.join(args.input_train, "train.csv"))
X_train = train_df.drop("target", axis=1)
y_train = train_df["target"]

# Train model
mlflow.start_run()
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
mlflow.log_param("n_estimators", 100)

# Save model
os.makedirs(args.output_model, exist_ok=True)
joblib.dump(model, os.path.join(args.output_model, "model.pkl"))
mlflow.end_run()
print("Model trained and saved")
'''

# Step 3 - Evaluation script
eval_script = '''
import argparse
import pandas as pd
from sklearn.metrics import accuracy_score
import mlflow
import joblib
import os

parser = argparse.ArgumentParser()
parser.add_argument("--input_test", type=str)
parser.add_argument("--input_model", type=str)
args = parser.parse_args()

# Load test data and model
test_df = pd.read_csv(os.path.join(args.input_test, "test.csv"))
X_test = test_df.drop("target", axis=1)
y_test = test_df["target"]
model = joblib.load(os.path.join(args.input_model, "model.pkl"))

# Evaluate
mlflow.start_run()
accuracy = accuracy_score(y_test, model.predict(X_test))
mlflow.log_metric("test_accuracy", accuracy)
mlflow.end_run()
print(f"Test accuracy: {accuracy}")
'''

# Write scripts to files
with open("pipeline_scripts/data_prep.py", "w") as f:
    f.write(data_prep_script)

with open("pipeline_scripts/train.py", "w") as f:
    f.write(train_script)

with open("pipeline_scripts/evaluate.py", "w") as f:
    f.write(eval_script)

print("Pipeline scripts created successfully")

Pipeline scripts created successfully


In [3]:
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

# Define each step as a command component
data_prep_component = command(
    name="data_prep",
    display_name="Data Preparation",
    code="pipeline_scripts",
    command="python data_prep.py --output_train ${{outputs.output_train}} --output_test ${{outputs.output_test}}",
    outputs={
        "output_train": Output(type=AssetTypes.URI_FOLDER),
        "output_test": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="ml-compute-md"
)

train_component = command(
    name="train_model",
    display_name="Train Model",
    code="pipeline_scripts",
    command="python train.py --input_train ${{inputs.input_train}} --output_model ${{outputs.output_model}}",
    inputs={
        "input_train": Input(type=AssetTypes.URI_FOLDER)
    },
    outputs={
        "output_model": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="ml-compute-md"
)

eval_component = command(
    name="evaluate_model",
    display_name="Evaluate Model",
    code="pipeline_scripts",
    command="python evaluate.py --input_test ${{inputs.input_test}} --input_model ${{inputs.input_model}}",
    inputs={
        "input_test": Input(type=AssetTypes.URI_FOLDER),
        "input_model": Input(type=AssetTypes.URI_FOLDER)
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="ml-compute-md"
)

# Define pipeline
@pipeline(
    display_name="Iris Training Pipeline",
    description="End to end ML pipeline - data prep, training, evaluation"
)
def iris_pipeline():
    data_prep_step = data_prep_component()
    
    train_step = train_component(
        input_train=data_prep_step.outputs.output_train
    )
    
    eval_step = eval_component(
        input_test=data_prep_step.outputs.output_test,
        input_model=train_step.outputs.output_model
    )

# Create and submit pipeline
pipeline_job = iris_pipeline()
pipeline_job.settings.default_compute = "ml-compute-md"

returned_job = ml_client.jobs.create_or_update(pipeline_job, experiment_name="iris-pipeline")
print(f"Pipeline submitted: {returned_job.name}")
print(f"View in studio: {returned_job.studio_url}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading pipeline_scripts (0.0 MBs): 

Pipeline submitted: loyal_cabbage_9xfn4f9cz6
View in studio: https://ml.azure.com/runs/loyal_cabbage_9xfn4f9cz6?wsid=/subscriptions/749d055e-0977-4255-acd4-54c54916bff8/resourcegroups/ml-learning-rg/workspaces/ml-learning-workspace&tid=5fb38fb2-979b-407e-8873-dec137663a6a
